In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .appName('pyspark-run-with-gcp-bucket2') \
    .config("spark.jars", "/home/kantundpeterpan/projects/zoomcamp/zcde_space/week5/gcs-connector-hadoop3-latest.jar") \
    .config("spark.sql.repl.eagerEval.enabled", True) \
    .getOrCreate()

25/03/06 14:39:38 WARN Utils: Your hostname, mystuff resolves to a loopback address: 127.0.1.1; using 193.168.147.155 instead (on interface eth0)
25/03/06 14:39:38 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
25/03/06 14:39:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [3]:
# Configure GCS authentication
spark.conf.set("google.cloud.auth.service.account.enable", "true")
spark._jsc.hadoopConfiguration().set("google.cloud.auth.service.account.json.keyfile", 
                                     "/home/kantundpeterpan/projects/zoomcamp/zcde_space/week1/3_intro_terraform/workspaceaddon-436615-4bcf737409b7.json")
spark._jsc.hadoopConfiguration().set('fs.gs.impl', 'com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem')

In [4]:
yellow_schema = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("gs://workspaceaddon-436615/yellow_tripdata_2021-01.csv.gz").schema

In [5]:
green_schema = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("gs://workspaceaddon-436615/green_tripdata_2021-01.csv.gz").schema

In [11]:
year = 2020
taxi = 'green'

schemas = {
    'yellow':yellow_schema,
    'green':green_schema
    }

for m in range(1,13):
    df = spark.read \
        .option("header", True) \
        .schema(schemas[taxi]) \
        .csv(f"gs://workspaceaddon-436615/{taxi}_tripdata_{year}-{m:02d}.csv")
    
    output_path = f"gs://workspaceaddon-436615/{taxi}/{year}/{m:02d}"
    print(output_path)
    
    df.repartition(4) \
        .write.parquet(output_path)

gs://workspaceaddon-436615/green/2020/01


gs://workspaceaddon-436615/green/2020/02


gs://workspaceaddon-436615/green/2020/03


gs://workspaceaddon-436615/green/2020/04


gs://workspaceaddon-436615/green/2020/05


...

gs://workspaceaddon-436615/green/2020/08


gs://workspaceaddon-436615/green/2020/09


gs://workspaceaddon-436615/green/2020/10


gs://workspaceaddon-436615/green/2020/11


gs://workspaceaddon-436615/green/2020/12


In [12]:
year = 2021
taxi = 'green'

schemas = {
    'yellow':yellow_schema,
    'green':green_schema
    }

for m in range(1,8):
    df = spark.read \
        .option("header", True) \
        .schema(schemas[taxi]) \
        .csv(f"gs://workspaceaddon-436615/{taxi}_tripdata_{year}-{m:02d}.csv.gz")
    
    output_path = f"gs://workspaceaddon-436615/{taxi}/{year}/{m:02d}"
    print(output_path)
    
    df.repartition(4) \
        .write.parquet(output_path)

gs://workspaceaddon-436615/green/2021/01


gs://workspaceaddon-436615/green/2021/02


gs://workspaceaddon-436615/green/2021/03


gs://workspaceaddon-436615/green/2021/04


gs://workspaceaddon-436615/green/2021/05


gs://workspaceaddon-436615/green/2021/06


gs://workspaceaddon-436615/green/2021/07


In [ ]:
spark.stop()